# mflux Text-to-Image — FLUX.1-schnell

Generate images with **[black-forest-labs/FLUX.1-schnell](https://huggingface.co/black-forest-labs/FLUX.1-schnell)** via **[mflux](https://github.com/filipstrand/mflux)** (Apple MLX / Metal).

**Not Hugging Face Diffusers.** Hub may still supply weights into `HF_HOME`; inference is mflux + MLX.

This Mac has **16 GB** unified memory. Full PyTorch/`diffusers` FLUX (~24 GB) OOMs or crashes the kernel. **mflux + 4-bit (~10 GB)** fits and runs on Metal.

Close other heavy apps before loading the model.


## Load env (`HF_HOME`, `HF_TOKEN`)

Reads the repo-root `.env` (see `.env.example`). Weights download into `HF_HOME` on first run so other projects can reuse them.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Repo-root .env when the notebook cwd is the project subfolder
env_path = Path.cwd().parent / ".env"
if not env_path.is_file():
    env_path = Path.cwd() / ".env"
load_dotenv(env_path)

if "HF_HOME" in os.environ:
    os.environ["HF_HOME"] = os.path.expanduser(os.environ["HF_HOME"])

# importing the constants after setting the env var.
from huggingface_hub.constants import HF_HOME, HF_HUB_CACHE

print("HF_HOME:", HF_HOME)
print("hub cache:", HF_HUB_CACHE)

## Load model (mflux, 4-bit)

Uses MLX on Apple Silicon. `quantize=4` is required on 16 GB machines. First load may take a while while weights are quantized.

In [ ]:
from IPython.display import display
from mflux.models.flux.variants.txt2img.flux import Flux1

# 4-bit ~10 GB — needed for 16 GB unified memory
QUANTIZE = 4
HEIGHT = 768
WIDTH = 768
STEPS = 4

flux = Flux1.from_name(model_name="schnell", quantize=QUANTIZE)

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("model: FLUX.1-schnell | quantize:", QUANTIZE, "| size:", WIDTH, "x", HEIGHT)

## Demo: human thinking — Book Illustration style

Generates a storybook-style illustration of a person deep in thought.

In [ ]:
prompt = (
    "A thoughtful human deep in thought, hand on chin, "
    "Book Illustration style, detailed ink and watercolor, storybook art"
)

result = flux.generate_image(
    seed=42,
    prompt=prompt,
    num_inference_steps=STEPS,
    height=HEIGHT,
    width=WIDTH,
)
image = result.image

out_path = OUTPUT_DIR / "human_thinking_book_illustration.png"
result.save(out_path, overwrite=True)
display(image)
print("saved:", out_path)

## January calendar page

Minimalist cartoon calendar header with a man sitting on a chair.

In [ ]:
custom_prompt = (
    "A minimalist cartoon illustration for a calendar page. "
    "At the top, the text clearly reads 'JANUARY'. "
    "Below the text, a clean vector drawing of a man sitting peacefully on a chair. "
    "Flat design, muted corporate pastel colors, line art, white background, highly legible layout."
)

result = flux.generate_image(
    seed=7,
    prompt=custom_prompt,
    num_inference_steps=STEPS,
    height=HEIGHT,
    width=WIDTH,
)
custom_image = result.image

custom_path = OUTPUT_DIR / "january_calendar_page.png"
result.save(custom_path, overwrite=True)
display(custom_image)
print("saved:", custom_path)